# Week 8-G: Measured hard-case and outcome-weighted generator adaptation

Compare original SARA with **matched-context uniform SFT**, **HC-LoRA**, and **OW-LoRA**. All three trainable variants use the same FIT examples and context allocation; only their loss weights differ. They start independently from the original trained SARA checkpoint.

New rank-8 residual LoRA factors target q/k/v/o projections. The original SARA adapter, base weights, and projector stay frozen and unmerged. Raw HC weights are 2 for helpful/distractor cases and 1 for neutral cases. OW uses `1 + 2 * max_delta` for helpful cases, `1 + 2 * (-min_delta)` for hurt-only cases, and 1 for neutral cases. Each variant divides weights by its FIT mean. Gamma=2, seed=42, learning rate=1e-4, and two epochs are fixed before validation.

FIT outcomes may select supervised training contexts: helpful cases promote the best FIT candidate; hurt-only/neutral cases retain canonical k4. Validation has **no oracle passage selection**. Report adapted direct k4 generation and promoted revisions under the unchanged frozen W8-E policy. The latter uses the original unadapted draft and returns that draft on STOP. INTERNAL-VAL is previously inspected development confirmation; TEST is unused.

**Set `RUN_FULL = True` below and Run All to start full training.** Otherwise Run All only checks inputs and the real GPU smoke. Full training is substantial GPU work across three independently trained variants. W8-A through W8-F are reused.

In [1]:
import sys, subprocess, json
from pathlib import Path
import pandas as pd
from IPython.display import display
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / 'configs/week8/w8_dafir.yaml').is_file())
SCRIPT = ROOT / 'scripts/week8/run_w8g_lora_training.py'
OUTPUT = ROOT / 'results/week8/w8g_lora'
RUN_FULL = True  # Set True to start full FIT training and validation.
EPOCHS = 2        # Fixed before validation; a different budget requires a new output directory.
def run(*args):
    subprocess.run([sys.executable, str(SCRIPT), *map(str,args)], cwd=ROOT, check=True)
print('Python:', sys.executable)
run('--check', '--output-dir', OUTPUT, '--epochs', EPOCHS)

Python: /home/gunavenkat/Downloads/CS787-RAG-Project/sara_env/bin/python
Verified real FIT/VAL surfaces, frozen policy, embeddings and paper disjointness: {'fit': 1746, 'internal_val': 342}


## Real GPU smoke

Train every variant on one real FIT example from each category, then evaluate two validation queries covering STOP and PROMOTE. Check exact identity before adaptation, nonzero new-adapter gradients, unchanged original SARA parameters, checkpoint saving, generation, and cumulative cost. Smoke scores are correctness checks, not scientific findings.

In [2]:
run('--smoke', '--output-dir', OUTPUT)
smoke = json.loads((OUTPUT / 'smoke/summary.json').read_text())
assert smoke['smoke_only'] and smoke['no_validation_selection'] and not smoke['TEST_used']
for proof in smoke['training_proofs'].values():
    assert proof['original_parameter_versions_unchanged'] and proof['only_new_LoRA_trainable']
    assert proof['zero_residual_identity_checks'] > 0
display(pd.read_csv(OUTPUT / 'smoke/comparison.csv'))

Verified completed measured W8-G run: /home/gunavenkat/Downloads/CS787-RAG-Project/results/week8/w8g_lora/smoke


,variant,context,queries,downstream_f1,gain_over_original_same_policy,help_rate_vs_original,hurt_rate_vs_original,mean_final_prompt_tokens,mean_cumulative_prompt_tokens,mean_estimated_cumulative_generation_latency_sec,generation_promote_rate,end_to_end_latency_sec
0,original_SARA,frozen_controller,2,0.633333,0.0,0.0,0.0,925.0,1456.5,3.731882,0.5,NaN
1,original_SARA,k4,2,0.633333,0.0,0.0,0.0,812.0,812.0,2.146233,0.0,NaN
2,uniform_SFT,frozen_controller,2,0.633333,0.0,0.0,0.0,925.0,1456.5,3.620029,0.5,NaN
3,uniform_SFT,k4,2,0.633333,0.0,0.0,0.0,812.0,812.0,2.095123,0.0,NaN
4,HC_LoRA,frozen_controller,2,0.633333,0.0,0.0,0.0,925.0,1456.5,3.650106,0.5,NaN
5,HC_LoRA,k4,2,0.633333,0.0,0.0,0.0,812.0,812.0,2.133462,0.0,NaN
6,OW_LoRA,frozen_controller,2,0.633333,0.0,0.0,0.0,925.0,1456.5,3.676181,0.5,NaN
7,OW_LoRA,k4,2,0.633333,0.0,0.0,0.0,812.0,812.0,2.155942,0.0,NaN


## Full fixed-budget experiment

Train on all 1,746 FIT queries and confirm on 342 INTERNAL-VAL queries. No validation-selected gamma, checkpoint, or epoch is reported. All variants are retained, including unfavorable results. The original SARA baseline uses verified saved real outputs; learned variants generate new answers. Fresh initialization and identity checks prevent trials from inheriting prior training.

Epoch checkpoints include new LoRA factors, optimizer state, source hashes and the original SARA checkpoint reference. They are additive checkpoints, not standalone merged generators. Interrupted training resumes at the last completed epoch; interrupted validation resumes per query/context. Completed runs verify hashes and reuse results. Different code/settings require a new experiment output directory.

In [3]:
if RUN_FULL:
    run('--full', '--output-dir', OUTPUT, '--epochs', EPOCHS)
    summary = json.loads((OUTPUT / 'full/summary.json').read_text())
    assert not summary['smoke_only'] and summary['no_validation_selection'] and not summary['TEST_used']
    display(pd.read_csv(OUTPUT / 'full/comparison.csv'))
else:
    print('Smoke complete. Set RUN_FULL=True and Run All to start the full experiment.')

`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards: 100%|██████████| 3/3 [00:00<00:00, 113.27it/s]
Some weights of XMistralForCausalLM were not initialized from the model checkpoint at mistralai/Mistral-7B-Instruct-v0.2 and are newly initialized: ['projector.projector.0.bias', 'projector.projector.0.weight', 'projector.projector.2.bias', 'projector.projector.2.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`
The new lm_head weights will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `

uniform_SFT epoch 1 {'query_mean_weighted_nll': 1.0852132653348139, 'query_mean_unweighted_nll': 1.0852132653348139, 'queries': 1746, 'only_new_adapter_gradients_verified': True}


uniform_SFT FIT epoch 2: 100%|██████████| 1746/1746 [3:08:46<00:00,  6.49s/it]  


uniform_SFT epoch 2 {'query_mean_weighted_nll': 0.8924277213139125, 'query_mean_unweighted_nll': 0.8924277213139125, 'queries': 1746, 'only_new_adapter_gradients_verified': True}


Loading checkpoint shards: 100%|██████████| 3/3 [00:00<00:00, 118.20it/s]
Some weights of XMistralForCausalLM were not initialized from the model checkpoint at mistralai/Mistral-7B-Instruct-v0.2 and are newly initialized: ['projector.projector.0.bias', 'projector.projector.0.weight', 'projector.projector.2.bias', 'projector.projector.2.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
HC_LoRA FIT epoch 2:   0%|          | 0/1746 [00:00<?, ?it/s]

...

OW_LoRA FIT epoch 2: 100%|██████████| 1746/1746 [3:20:15<00:00,  6.88s/it]  


OW_LoRA epoch 2 {'query_mean_weighted_nll': 0.8331595109298267, 'query_mean_unweighted_nll': 0.9092086678327361, 'queries': 1746, 'only_new_adapter_gradients_verified': True}


OW_LoRA measured validation: 100%|██████████| 342/342 [44:47<00:00,  7.86s/it]


      variant           context  queries  downstream_f1  gain_over_original_same_policy  help_rate_vs_original  hurt_rate_vs_original  mean_final_prompt_tokens  mean_cumulative_prompt_tokens  mean_estimated_cumulative_generation_latency_sec  generation_promote_rate end_to_end_latency_sec
original_SARA frozen_controller      342       0.400080                        0.000000               0.000000               0.000000               1017.593567                    1416.111111                                          3.931297                 0.421053                   None
original_SARA                k4      342       0.389061                        0.000000               0.000000               0.000000                935.157895                     935.157895                                          2.691779                 0.000000                   None
  uniform_SFT frozen_controller      342       0.412340                        0.012260               0.087719               0.055556

,variant,context,queries,downstream_f1,gain_over_original_same_policy,help_rate_vs_original,hurt_rate_vs_original,mean_final_prompt_tokens,mean_cumulative_prompt_tokens,mean_estimated_cumulative_generation_latency_sec,generation_promote_rate,end_to_end_latency_sec
0,original_SARA,frozen_controller,342,0.400080,0.000000,0.000000,0.000000,1017.593567,1416.111111,3.931297,0.421053,NaN
1,original_SARA,k4,342,0.389061,0.000000,0.000000,0.000000,935.157895,935.157895,2.691779,0.000000,NaN
2,uniform_SFT,frozen_controller,342,0.412340,0.012260,0.087719,0.055556,1017.593567,1416.111111,4.046358,0.421053,NaN
3,uniform_SFT,k4,342,0.406722,0.017660,0.198830,0.160819,935.157895,935.157895,2.887133,0.000000,NaN
4,HC_LoRA,frozen_controller,342,0.409215,0.009135,0.087719,0.061404,1017.593567,1416.111111,4.151086,0.421053,NaN
5,HC_LoRA,k4,342,0.415027,0.025966,0.213450,0.152047,935.157895,935.157895,3.046624,0.000000,NaN
6,OW_LoRA,frozen_controller,342,0.403553,0.003473,0.073099,0.058480,1017.593567,1416.111111,4.097550,0.421053,NaN
7,OW_LoRA,k4,342,0.403986,0.014925,0.195906,0.140351,935.157895,935.157895,2.966230,0.000000,NaN


## Interpret the two evaluations separately

`k4` measures direct generation with adapted SARA. `frozen_controller` keeps the original W8-E gate, candidate identities and base draft fixed; only promoted revision generations use the new adapter. It does not reroute using changed drafts or replay old promotions as adapted outcomes. Helpful/harmful rates compare each variant with original SARA under the same fixed policy. Harmful outputs are retained.

Use generated-answer F1 as the primary quality measure. Compare HC/OW against the matched-context uniform control to assess the weighting, and use paired paper comparisons when interpreting full results. Weighted training NLL differs in its objective; unweighted query-mean NLL is saved separately.

Cumulative prompt cost counts both original draft and promoted revision. New pass timing is measured, while the original-draft timing comes from saved real records. These components are not measured end-to-end deployment latency. Existing W8-E and W8-F models/results remain unchanged.

In [4]:
folder = OUTPUT / ('full' if RUN_FULL else 'smoke')
print('Results:', folder / 'comparison.csv')
print('Protocol:', folder / 'protocol_frozen_before_validation.json')
print('Provenance:', folder / 'provenance.json')
print('Additive LoRA checkpoints:', list(folder.glob('*.pt')))
print('Training example/weight manifests:', list(folder.glob('*_fit_examples.json')))
print('Measured validation predictions:', list(folder.glob('*_predictions.jsonl')))

Results: /home/gunavenkat/Downloads/CS787-RAG-Project/results/week8/w8g_lora/full/comparison.csv
Protocol: /home/gunavenkat/Downloads/CS787-RAG-Project/results/week8/w8g_lora/full/protocol_frozen_before_validation.json
Provenance: /home/gunavenkat/Downloads/CS787-RAG-Project/results/week8/w8g_lora/full/provenance.json
Additive LoRA checkpoints: [PosixPath('/home/gunavenkat/Downloads/CS787-RAG-Project/results/week8/w8g_lora/full/OW_LoRA.pt'), PosixPath('/home/gunavenkat/Downloads/CS787-RAG-Project/results/week8/w8g_lora/full/uniform_SFT.pt'), PosixPath('/home/gunavenkat/Downloads/CS787-RAG-Project/results/week8/w8g_lora/full/HC_LoRA.pt')]
Training example/weight manifests: [PosixPath('/home/gunavenkat/Downloads/CS787-RAG-Project/results/week8/w8g_lora/full/OW_LoRA_fit_examples.json'), PosixPath('/home/gunavenkat/Downloads/CS787-RAG-Project/results/week8/w8g_lora/full/uniform_SFT_fit_examples.json'), PosixPath('/home/gunavenkat/Downloads/CS787-RAG-Project/results/week8/w8g_lora/full/HC_L